# Logistic Regression — SHAP Error Analysis

This notebook reconstructs the explainability and error-analysis stage of the official Logistic Regression experiment (original Steps 7A–8G).

It uses the **locked official baseline** and does not retrain the model or refit the TF-IDF vocabulary.

Main objectives:

1. explain all **88 official False Positive** predictions with SHAP;
2. rank features by mean absolute SHAP magnitude;
3. inspect signed SHAP direction;
4. compare False Positive patterns with correctly classified legitimate emails;
5. analyse contextual/token patterns;
6. inspect all **48 False Negative** cases as safety context.

Run `01_dataset_preparation.ipynb` and `02_official_baseline.ipynb` first in the same runtime.


In [ ]:
# Install SHAP if necessary.
!pip -q install shap

import os
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix

EXPECTED_TP = 4660
EXPECTED_FP = 88
EXPECTED_FN = 48
EXPECTED_TN = 5920
EXPECTED_FEATURES = 10000

print("SHAP error-analysis configuration loaded.")


## 1. Verify the locked official baseline

No model training occurs in this notebook. The exact official predictions and locked TF-IDF representation must already exist.


In [ ]:
required_objects = [
    "lr_model",
    "tfidf_vectorizer",
    "X_train_tfidf",
    "X_test_tfidf",
    "X_train",
    "X_test",
    "y_train",
    "y_test",
    "y_pred",
    "y_prob"
]

missing = [name for name in required_objects if name not in globals()]
if missing:
    raise RuntimeError(
        "Missing required objects: "
        + ", ".join(missing)
        + "\nRun notebooks 01 and 02 first in the same runtime."
    )

cm = confusion_matrix(y_test, y_pred)
assert np.array_equal(cm, np.array([[4660, 88], [48, 5920]]))
assert X_train_tfidf.shape == (42861, 10000)
assert X_test_tfidf.shape == (10716, 10000)

feature_names = tfidf_vectorizer.get_feature_names_out()
assert len(feature_names) == EXPECTED_FEATURES

print("Locked official baseline verified.")
print(cm)


## 2. Identify official prediction groups

Under the thesis convention:

- False Positive = legitimate (`0`) predicted phishing (`1`);
- False Negative = phishing (`1`) predicted legitimate (`0`);
- True Positive = legitimate (`0`) correctly predicted legitimate (`0`).

All 88 False Positives are analysed rather than selecting isolated examples.


In [ ]:
y_test_array = np.asarray(y_test)
y_pred_array = np.asarray(y_pred)

fp_positions = np.where((y_test_array == 0) & (y_pred_array == 1))[0]
fn_positions = np.where((y_test_array == 1) & (y_pred_array == 0))[0]
tp_positions = np.where((y_test_array == 0) & (y_pred_array == 0))[0]

assert len(fp_positions) == 88
assert len(fn_positions) == 48
assert len(tp_positions) == 4660

X_fp_tfidf = X_test_tfidf[fp_positions]
X_fn_tfidf = X_test_tfidf[fn_positions]
X_tp_tfidf = X_test_tfidf[tp_positions]

print("False Positives:", len(fp_positions))
print("False Negatives:", len(fn_positions))
print("Correct Legitimate (TP):", len(tp_positions))


## 3. SHAP LinearExplainer

The original analysis used `shap.LinearExplainer` with the locked Logistic Regression model and training TF-IDF representation. The SHAP installation may internally subsample the background for efficiency; the model, vocabulary and test cases remain unchanged.


In [ ]:
explainer = shap.LinearExplainer(
    lr_model,
    X_train_tfidf
)

print("SHAP LinearExplainer ready.")


## 4. SHAP values for all 88 False Positives

The resulting matrix contains one SHAP contribution for every TF-IDF feature in every official False Positive email.


In [ ]:
shap_fp = explainer(X_fp_tfidf)

shap_fp_values = np.asarray(shap_fp.values)

print("Number of False Positives analysed:", len(fp_positions))
print("Number of TF-IDF features:", len(feature_names))
print("SHAP Values Shape:", shap_fp_values.shape)

assert shap_fp_values.shape == (88, 10000)

print("False Positive SHAP analysis verified.")


## 5. Mean absolute SHAP feature importance

For each of the 10,000 features, the absolute SHAP contributions are averaged across all 88 False Positive emails. This measures average **influence magnitude**, regardless of direction.

The top 20 values below are the thesis checkpoint for this stage.


In [ ]:
mean_abs_shap_fp = np.mean(np.abs(shap_fp_values), axis=0)

fp_importance = pd.DataFrame({
    "Feature": feature_names,
    "Mean_Absolute_SHAP": mean_abs_shap_fp
}).sort_values(
    "Mean_Absolute_SHAP",
    ascending=False
).reset_index(drop=True)

fp_top20 = fp_importance.head(20).copy()
fp_top20.index = np.arange(1, 21)

display(fp_top20)


In [ ]:
expected_fp_top20 = [
    ("escapenumber", 0.190372),
    ("escapenumber escapenumber", 0.121225),
    ("wrote", 0.110621),
    ("debian", 0.108820),
    ("escapelong escapelong", 0.096440),
    ("org", 0.093081),
    ("list", 0.087479),
    ("perl", 0.067060),
    ("thanks", 0.066060),
    ("cnn", 0.061555),
    ("com", 0.061189),
    ("help", 0.056937),
    ("price", 0.056565),
    ("lists", 0.051712),
    ("speakup", 0.048426),
    ("cabot", 0.046147),
    ("code", 0.045514),
    ("net", 0.044974),
    ("use", 0.040982),
    ("unsubscribe", 0.040191)
]

print("Recorded thesis checkpoint:")
display(pd.DataFrame(
    expected_fp_top20,
    columns=["Feature", "Recorded_Mean_Absolute_SHAP"]
))

# SHAP package versions can differ slightly in background handling.
# We therefore verify the recorded ranking separately rather than
# silently replacing the values generated by the current environment.
generated_top_features = fp_top20["Feature"].tolist()
recorded_top_features = [x[0] for x in expected_fp_top20]

print("Exact top-20 feature order reproduced:",
      generated_top_features == recorded_top_features)


## 6. Reproduce the False Positive SHAP feature-importance figure

This regenerates the analysis figure from the current run. The repository should retain the original experiment figure as the authoritative thesis artifact when available.


In [ ]:
plot_data = fp_top20.sort_values("Mean_Absolute_SHAP")

plt.figure(figsize=(9, 7))
plt.barh(plot_data["Feature"], plot_data["Mean_Absolute_SHAP"])
plt.xlabel("Mean Absolute SHAP Value")
plt.ylabel("Feature")
plt.title("SHAP Feature Importance for Logistic Regression False Positive Predictions")
plt.tight_layout()
plt.show()


## 7. Signed SHAP direction

Mean absolute SHAP identifies influential features but removes direction. Signed SHAP values are therefore inspected separately to determine whether a feature tends to push a prediction toward phishing or toward legitimate classification.

Positive SHAP values push the model output toward class `1` (phishing); negative values push away from phishing.


In [ ]:
mean_signed_shap_fp = np.mean(shap_fp_values, axis=0)

signed_fp = pd.DataFrame({
    "Feature": feature_names,
    "Mean_Signed_SHAP": mean_signed_shap_fp
})

print("Strongest mean positive SHAP contributions:")
display(
    signed_fp.sort_values(
        "Mean_Signed_SHAP", ascending=False
    ).head(20)
)

print("\nStrongest mean negative SHAP contributions:")
display(
    signed_fp.sort_values(
        "Mean_Signed_SHAP", ascending=True
    ).head(20)
)


Recorded examples from the original signed analysis included positive contributions for `debian`, `escapenumber`, and `list`, and negative contributions for `price` and `cabot`. These examples illustrate why magnitude and direction must be interpreted separately.


In [ ]:
recorded_signed_examples = pd.DataFrame({
    "Feature": ["debian", "escapenumber", "list", "price", "cabot"],
    "Recorded_Mean_Signed_SHAP": [
        0.103805,
        0.090321,
        0.083745,
        -0.048884,
        -0.046147
    ]
})
display(recorded_signed_examples)


## 8. Comparison with correctly classified legitimate emails

The original experiment also analysed all **4,660 correctly classified legitimate emails**. This provides context: a feature appearing in a False Positive is not automatically a phishing-specific signal.


In [ ]:
shap_tp = explainer(X_tp_tfidf)
shap_tp_values = np.asarray(shap_tp.values)

assert shap_tp_values.shape == (4660, 10000)

mean_abs_shap_tp = np.mean(np.abs(shap_tp_values), axis=0)

tp_importance = pd.DataFrame({
    "Feature": feature_names,
    "Mean_Absolute_SHAP": mean_abs_shap_tp
}).sort_values(
    "Mean_Absolute_SHAP",
    ascending=False
).reset_index(drop=True)

tp_top20 = tp_importance.head(20).copy()
tp_top20.index = np.arange(1, 21)

display(tp_top20)

print("\nRecorded leading TP features:")
print("perl        ≈ 0.240419")
print("escapenumber≈ 0.233550")
print("wrote       ≈ 0.211399")
print("debian      ≈ 0.190586")


In [ ]:
plot_tp = tp_top20.sort_values("Mean_Absolute_SHAP")

plt.figure(figsize=(9, 7))
plt.barh(plot_tp["Feature"], plot_tp["Mean_Absolute_SHAP"])
plt.xlabel("Mean Absolute SHAP Value")
plt.ylabel("Feature")
plt.title("SHAP Feature Importance for Correctly Classified Legitimate Emails")
plt.tight_layout()
plt.show()


## 9. Context/token frequency comparison

The original contextual analysis counted selected strings in the cleaned text of False Positive and correctly classified legitimate emails.

These counts are descriptive evidence; they are not new model features in this notebook.


In [ ]:
test_text_array = np.asarray(X_test)

fp_texts = pd.Series(test_text_array[fp_positions])
tp_texts = pd.Series(test_text_array[tp_positions])

context_tokens = ["com", "use", "net", "thanks", "price", "cabot"]

def substring_frequency(text_series, token):
    return int(text_series.str.contains(
        token, case=False, regex=False
    ).sum())

context_rows = []
for token in context_tokens:
    fp_count = substring_frequency(fp_texts, token)
    tp_count = substring_frequency(tp_texts, token)
    context_rows.append({
        "Token": token,
        "FP_Count": fp_count,
        "FP_Percent": 100 * fp_count / len(fp_texts),
        "TP_Count": tp_count,
        "TP_Percent": 100 * tp_count / len(tp_texts)
    })

context_frequency = pd.DataFrame(context_rows)
display(context_frequency)


Recorded checkpoint from the original context analysis:

| Token | FP count | FP % | Correct legitimate count | Correct legitimate % |
|---|---:|---:|---:|---:|
| com | 57 | 64.77 | 3748 | 80.43 |
| use | 43 | 48.86 | 2686 | 57.64 |
| net | 29 | 32.95 | 1122 | 24.08 |
| thanks | 17 | 19.32 | 1118 | 23.99 |
| price | 9 | 10.23 | 204 | 4.38 |
| cabot | 3 | 3.41 | 12 | 0.26 |


## 10. URL and message-pattern analysis of False Positives

The original Step 8 analysis also inspected recurring textual patterns in the 88 False Positive emails.


In [ ]:
pattern_terms = ["http", "www", "attach", "browser", "unsubscribe"]

pattern_rows = []
for term in pattern_terms:
    count = substring_frequency(fp_texts, term)
    pattern_rows.append({
        "Pattern": term,
        "FP_Count": count,
        "FP_Percent": 100 * count / len(fp_texts)
    })

fp_pattern_frequency = pd.DataFrame(pattern_rows)
display(fp_pattern_frequency)

print("\nRecorded original counts:")
print("http=40 | www=30 | attach=7 | browser=5 | unsubscribe=11")


## 11. SHAP analysis of all 48 False Negatives

False Negative cases are phishing emails incorrectly classified as legitimate. They are analysed as safety context because the thesis objective requires reducing False Positives **without increasing False Negatives**.


In [ ]:
shap_fn = explainer(X_fn_tfidf)
shap_fn_values = np.asarray(shap_fn.values)

assert shap_fn_values.shape == (48, 10000)

mean_abs_shap_fn = np.mean(np.abs(shap_fn_values), axis=0)

fn_importance = pd.DataFrame({
    "Feature": feature_names,
    "Mean_Absolute_SHAP": mean_abs_shap_fn
}).sort_values(
    "Mean_Absolute_SHAP",
    ascending=False
).reset_index(drop=True)

print("False Negative SHAP matrix:", shap_fn_values.shape)
display(fn_importance.head(20))


## 12. Context report used for refinement

The original analysis inspected contextual appearances of the features `debian`, `perl`, `org`, `list`, and `cnn`. The report is reproduced below from the locked test texts and can be exported for later review.


In [ ]:
focus_terms = ["debian", "perl", "org", "list", "cnn"]

context_report_rows = []

for pos in fp_positions:
    text = str(test_text_array[pos])
    matched = [
        term for term in focus_terms
        if term.lower() in text.lower()
    ]
    if matched:
        context_report_rows.append({
            "Test_Position": int(pos),
            "Original_Index": int(X_test.index[pos]),
            "Matched_Features": ", ".join(matched),
            "Text": text
        })

context_report = pd.DataFrame(context_report_rows)

print("Context-report rows:", len(context_report))
display(context_report.head(20))


## 13. Save reproducible analysis tables

These tables summarize the explainability evidence and feed the later counterfactual/refinement stages. No retraining occurs here.


In [ ]:
OUTPUT_FOLDER = "shap_error_analysis_outputs"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

fp_top20.to_csv(
    os.path.join(OUTPUT_FOLDER, "shap_fp_top20.csv"),
    index=True
)
tp_top20.to_csv(
    os.path.join(OUTPUT_FOLDER, "shap_tp_top20.csv"),
    index=True
)
fn_importance.head(20).to_csv(
    os.path.join(OUTPUT_FOLDER, "shap_fn_top20.csv"),
    index=False
)
context_frequency.to_csv(
    os.path.join(OUTPUT_FOLDER, "context_frequency_fp_vs_tp.csv"),
    index=False
)
fp_pattern_frequency.to_csv(
    os.path.join(OUTPUT_FOLDER, "fp_url_message_patterns.csv"),
    index=False
)
context_report.to_csv(
    os.path.join(OUTPUT_FOLDER, "context_report.csv"),
    index=False
)

summary = pd.DataFrame([
    {"Analysis": "Official False Positives", "Count": 88},
    {"Analysis": "Correct Legitimate", "Count": 4660},
    {"Analysis": "Official False Negatives", "Count": 48},
    {"Analysis": "TF-IDF Features", "Count": 10000}
])

summary.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "shap_guided_contextual_refinement_summary.csv"
    ),
    index=False
)

display(summary)
print("Analysis tables saved to:", OUTPUT_FOLDER)


## SHAP error-analysis checkpoint

This notebook establishes the explainability evidence behind the later refinement:

**Official baseline → False Positive error analysis → SHAP feature analysis → contextual comparison → False Negative safety context → counterfactual investigation**

No corrective model is trained in this notebook. The next core notebook is `04_counterfactual_analysis.ipynb`.
